## Gold — `fato_obras` (star schema CNO)

**Origem:** `silver.cno` x `silver.cno_areas` + dimensões → **Destino:** `workspace.gold.fato_obras`

- **Modelo:** Star Schema.
  - **Fato:** `fato_obras` — medidas `area_total`, `metragem`; dimensão degenerada `unidade_de_medida`; NK `cno`.
  - **Dimensões:** `dim_data` (role-playing: `sk_data_inicio` e `sk_data_situacao`), `dim_situacao`, `dim_municipio`, `dim_area`.
- **Grão:** 1 linha por obra x área declarada — o registro da obra se **repete para cada área/tipo** de `cno_areas` (inner join por `cno`).
- **Transformações:**
  - Mantém somente obras com `data_de_inicio` a partir de **1990-01-01** (alinhado ao intervalo da `dim_data`).
  - `sk_data_*` = inteiro AAAAMMDD determinístico (mesma regra da `gold.dim_data` — dispensa lookup).
  - Lookups de SK: `dim_situacao` (por código), `dim_municipio` (por **`codigo_tom`** — o CNO usa código TOM/SIAFI) e `dim_area` (join **null-safe** na combinação dos 5 atributos textuais).
  - Ordem das colunas: `cno`, depois as datas logo no início (cada uma ao lado da sua SK) e demais FKs/medidas em seguida.
- **Linhagem:** CSV dados.gov.br → bronze → silver (`cno`, `cno_areas`) + `silver.municipios` → gold (`dim_*`, `fato_obras`).

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from catalogo.cadastro_nacional_obras import FATO_OBRAS_COMMENTS

In [0]:
SILVER_CNO = "workspace.silver.cno"
SILVER_CNO_AREAS = "workspace.silver.cno_areas"
DIM_DATA = "workspace.gold.dim_data"
DIM_SITUACAO = "workspace.gold.dim_situacao"
DIM_MUNICIPIO = "workspace.gold.dim_municipio"
DIM_AREA = "workspace.gold.dim_area"
TARGET_TABLE = "workspace.gold.fato_obras"

# Corte alinhado ao início do calendário da dim_data
DATA_INICIO_CORTE = "1990-01-01"

# Contrato de saída gold.fato_obras:
# datas logo no início, após o código do cno, cada uma junto da sua SK
COLUNAS_ORDENADAS = [
    "cno",
    "sk_data_inicio",
    "data_de_inicio",
    "sk_data_situacao",
    "data_da_situacao",
    "sk_situacao",
    "sk_municipio",
    "sk_area",
    "unidade_de_medida",
    "area_total",
    "metragem",
]

In [0]:
df_cno = spark.table(SILVER_CNO)
print(f"Silver cno: {df_cno.count():,} obras")

# Somente obras iniciadas a partir de 1990
df_cno = df_cno.filter(F.col("data_de_inicio") >= F.to_date(F.lit(DATA_INICIO_CORTE)))
print(f"Obras com data_de_inicio >= {DATA_INICIO_CORTE}: {df_cno.count():,}")

# Repete o registro da obra para cada área declarada (grão: obra x área)
df_areas = spark.table(SILVER_CNO_AREAS)
print(f"Silver cno_areas: {df_areas.count():,} áreas")

df = df_cno.join(df_areas, on="cno", how="inner")
print(f"Fato antes dos lookups (obras x áreas): {df.count():,} linhas")
display(df.limit(5))

In [0]:
# Role-playing de dim_data: SK AAAAMMDD determinística (mesma regra da geração do calendário)
df = (
    df.withColumn("sk_data_inicio", F.date_format("data_de_inicio", "yyyyMMdd").cast("int"))
    .withColumn("sk_data_situacao", F.date_format("data_da_situacao", "yyyyMMdd").cast("int"))
)

# dim_situacao: código -> SK
df_situacao = spark.table(DIM_SITUACAO).select(
    F.col("codigo_situacao").alias("situacao"),
    F.col("sk_situacao"),
)
df = df.join(df_situacao, on="situacao", how="left")

# dim_municipio: código TOM (SIAFI) -> SK; silver.cno já armazena o código normalizado com pad de 4 dígitos
df_municipio = spark.table(DIM_MUNICIPIO).select(
    F.lpad(F.col("codigo_tom"), 4, "0").alias("codigo_do_municipio"),
    F.col("sk_municipio"),
)
df = df.join(df_municipio, on="codigo_do_municipio", how="left")

# dim_area: combinação dos 5 códigos -> SK (null-safe pois tipo_de_area_complementar pode ser nulo)
COMBINACAO = [
    "categoria",
    "destinacao",
    "tipo_de_obra",
    "tipo_de_area",
    "tipo_de_area_complementar",
]
df_area = spark.table(DIM_AREA)
condicao_area = df["categoria"].eqNullSafe(df_area["categoria"])
for coluna in COMBINACAO[1:]:
    condicao_area = condicao_area & df[coluna].eqNullSafe(df_area[coluna])

df = df.join(
    df_area.select(*COMBINACAO, "sk_area"),
    on=condicao_area,
    how="left",
).drop(df_area["categoria"], df_area["destinacao"], df_area["tipo_de_obra"], df_area["tipo_de_area"], df_area["tipo_de_area_complementar"])

df = df.select(*COLUNAS_ORDENADAS)
print(f"Fato final: {df.count():,} linhas")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    FATO_OBRAS_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
cnos = spark.table(TARGET_TABLE).select("cno").distinct().count()

# Integridade referencial: nenhuma SK pode ficar nula
fks = ["sk_data_inicio", "sk_data_situacao", "sk_situacao", "sk_municipio", "sk_area"]
nulos = {fk: spark.table(TARGET_TABLE).filter(F.col(fk).isNull()).count() for fk in fks}
print(f"Total: {total:,} linhas | Obras distintas: {cnos:,} | Áreas por obra: {round(total / cnos, 2)}")
print(f"FKs nulas: {nulos}")
assert all(v == 0 for v in nulos.values()), "FK nula em fato_obras"

# Cobertura das datas dentro do calendário da dim_data
display(spark.sql(f"SELECT min(data_de_inicio) AS primeira, max(data_de_inicio) AS ultima FROM {TARGET_TABLE}"))
display(spark.sql(f"SELECT d.descricao, count(*) AS qtd_linhas_fato FROM {TARGET_TABLE} f JOIN {DIM_SITUACAO} d ON f.sk_situacao = d.sk_situacao GROUP BY d.descricao ORDER BY qtd_linhas_fato DESC"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))